In [12]:
print("CSC 546 MinHash and LSH Project")
print("Jupyter Notebook is working!")

CSC 546 MinHash and LSH Project
Jupyter Notebook is working!


In [13]:
import pandas as pd
from pathlib import Path

# Find your project folder
cwd = Path.cwd()
project_root = cwd if (cwd / "data").exists() else cwd.parent

# Your downloaded Amazon review file
file_path = project_root / "data" / "All_Beauty (1).jsonl"

# Load first 10,000 reviews
df = pd.read_json(
    file_path,
    lines=True,
    nrows=100000
)

print("Dataset loaded successfully!")
print("Number of reviews:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Number of reviews: 100000
Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True


In [14]:
# Data Cleaning
# Check how many reviews we loaded
print("Total reviews:", len(df))

# Check if any reviews are missing text
missing_reviews = df["text"].isna().sum()
print("Missing reviews:", missing_reviews)

# Count the number of words in each review
df["word_count"] = df["text"].fillna("").str.split().str.len()

# Look at the review length statistics
print("\nReview length statistics:")
print(df["word_count"].describe())

Total reviews: 100000
Missing reviews: 0

Review length statistics:
count    100000.000000
mean         38.424770
std          54.400559
min           0.000000
25%           9.000000
50%          22.000000
75%          46.000000
max        1531.000000
Name: word_count, dtype: float64


In [15]:
# Data Cleaning Pt2
#Goal: Clean the review text by removing punctuation, converting to lowercase, and removing extra spaces.
#Why: Cleaning is important because MinHash will compare groups of words. 
#We don't want punctuation or capitalization to make otherwise similar reviews appear different.
import re

# Function to clean the review text
def clean_text(text):

    # Check if the review is missing
    if not isinstance(text, str):
        return ""

    # Convert everything to lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Apply the function to all reviews
df["clean_text"] = df["text"].apply(clean_text)

# Compare the original and cleaned reviews
display(df[["text", "clean_text"]].head(10))

,text,clean_text
0,This spray is really nice. It smells really go...,this spray is really nice it smells really goo...
1,"This product does what I need it to do, I just...",this product does what i need it to do i just ...
2,"Smells good, feels great!",smells good feels great
3,Felt synthetic,felt synthetic
4,Love it,love it
5,The polish was quiet thick and did not apply s...,the polish was quiet thick and did not apply s...
6,Great for many tasks. I purchased these for m...,great for many tasks i purchased these for mak...
7,These were lightweight and soft but much too s...,these were lightweight and soft but much too s...
8,This is perfect for my between salon visits. I...,this is perfect for my between salon visits i ...
9,I get Keratin treatments at the salon at least...,i get keratin treatments at the salon at least...


In [16]:
# Remove reviews that are too short
# Count words after cleaning
df["clean_word_count"] = df["clean_text"].str.split().str.len()

# Keep reviews with at least 2 words
df_clean = df[df["clean_word_count"] >= 2].copy()

# Check how many reviews were removed
print("Reviews before cleaning:", len(df))
print("Reviews after cleaning:", len(df_clean))
print("Reviews removed:", len(df) - len(df_clean))

# Display some cleaned reviews
display(df_clean[["clean_text", "clean_word_count"]].head(10))

# Test our cleaning function
test_review = "This PRODUCT is REALLY good!!!"

print("Original:", test_review)
print("Cleaned:", clean_text(test_review))

Reviews before cleaning: 100000
Reviews after cleaning: 97579
Reviews removed: 2421


,clean_text,clean_word_count
0,this spray is really nice it smells really goo...,62
1,this product does what i need it to do i just ...,47
2,smells good feels great,4
3,felt synthetic,2
4,love it,2
5,the polish was quiet thick and did not apply s...,24
6,great for many tasks i purchased these for mak...,22
7,these were lightweight and soft but much too s...,32
8,this is perfect for my between salon visits i ...,72
9,i get keratin treatments at the salon at least...,89


Original: This PRODUCT is REALLY good!!!
Cleaned: this product is really good


In [ ]:
#Shingle Creation
# Create word shingles from our cleaned Amazon reviews.
# Shingles are groups of consecutive words that help us compare how similar
# two reviews are. We are starting with 2-word shingles because they keep
# some word order while still allowing us to find reviews with similar wording.
# These shingles will later be used for Jaccard similarity, MinHash, and LSH.


# Create word shingles from a review
def create_shingles(text, k=2):
    words = text.split()
    shingles = set()

    # Get each group of k consecutive words
    for i in range(len(words) - k + 1):
        shingle = tuple(words[i:i + k])
        shingles.add(shingle)

    return shingles


# Test with a sample review
review = "this product works really well"

print("2-word shingles:")
print(create_shingles(review, k=2))

print("\n3-word shingles:")
print(create_shingles(review, k=3))

# Create 2-word shingles for all cleaned reviews
df_clean["shingles"] = df_clean["clean_text"].apply(
    lambda text: create_shingles(text, k=2)
)

# Check the first review
print(df_clean["clean_text"].iloc[0])

print("\nShingles:")
print(df_clean["shingles"].iloc[0])

2-word shingles:
{('works', 'really'), ('really', 'well'), ('product', 'works'), ('this', 'product')}

3-word shingles:
{('works', 'really', 'well'), ('product', 'works', 'really'), ('this', 'product', 'works')}
this spray is really nice it smells really good goes on really fine and does the trick i will say it feels like you need a lot of it though to get the texture i want i have a lot of hair medium thickness i am comparing to other brands with yucky chemicals so i m gonna stick with this try it

Shingles:
{('m', 'gonna'), ('lot', 'of'), ('i', 'have'), ('really', 'nice'), ('like', 'you'), ('to', 'other'), ('it', 'though'), ('i', 'am'), ('will', 'say'), ('nice', 'it'), ('to', 'get'), ('say', 'it'), ('the', 'texture'), ('yucky', 'chemicals'), ('with', 'this'), ('the', 'trick'), ('chemicals', 'so'), ('spray', 'is'), ('get', 'the'), ('thickness', 'i'), ('comparing', 'to'), ('so', 'i'), ('goes', 'on'), ('though', 'to'), ('of', 'hair'), ('really', 'fine'), ('have', 'a'), ('need', 'a'), ('